## Data Wrangling - Extension (Label Encoding Original Dataset to perform SHAP analysis on each feature)

### We need to modify our original dataset that we fed into our selected Catboost Model to evaluate the influence of each feature towards smoking prediction

In [6]:
#import dataset that was used to build catboost model, which was selected as the successful model for smoking 
#prediction at the end of Part 1 of smoker prediction

### 1. Data import from Feature Engineering Notebook
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
# Load training/ test data
#original data from data/modeling directory
# Load when needed
with open('../data/modeling/data_splits.pkl', 'rb') as f:
    data = pickle.load(f)
    X_train = data['X_train']
    X_test = data['X_test']
    y_train = data['y_train']
    y_test = data['y_test']

In [7]:
# Get category and object columns
cat_obj_features = [col for col in X_train.columns if X_train[col].dtype.name in ['category', 'object']]

ordinal_features = []
nomial_features = []

for col in cat_obj_features:
    unique_vals = X_train[col].unique()
    print(f"{col}: {unique_vals}")

    # Check if the category is ordered
    if X_train[col].dtype.name == 'category' and X_train[col].cat.ordered:
        ordinal_features.append(col)
    else:
        nomial_features.append(col)

print("Ordinal features:", ordinal_features)
print("Nomial features:", nomial_features)

DISPCODE: [1100, 1200]
Categories (2, int64): [1100, 1200]
GENHLTH: [1, 2, 3, 4, 7, 5]
Categories (6, int64): [1 < 2 < 3 < 4 < 5 < 7]
PHYSHLTH: [1, 2, 3, 4]
Categories (4, int64): [1 < 2 < 3 < 4]
MENTHLTH: [1, 2, 3, 4]
Categories (4, int64): [1 < 2 < 3 < 4]
HLTHPLN1: [1, 7, 2]
Categories (3, int64): [1, 2, 7]
PERSDOC2: [1, 2, 7, 3]
Categories (4, int64): [1, 2, 3, 7]
MEDCOST: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHECKUP1: [4, 1, 2, 3, NaN]
Categories (4, int64): [1 < 2 < 3 < 4]
CVDINFR4: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CVDCRHD4: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CVDSTRK3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
ASTHMA3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCSCNCR: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCOCNCR: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCCOPD1: [2, 7, 1]
Categories (3, int64): [1, 2, 7]
HAVARTH3: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
ADDEPEV2: [2, 1, 7]
Categories (3, int64): [1, 2, 7]
CHCKIDNY: [2, 1, 7]
Categories (3, 

In [8]:
# 1. Create copies of the dataframes
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()
y_train_clean = y_train.copy()
y_test_clean = y_test.copy()

# 2. Drop records with 7 or 9 in any feature columns
def drop_7_9(df):
    mask = ~df.apply(lambda row: row.isin([7, 9]).any(), axis=1)
    return mask

train_mask = drop_7_9(X_train_clean)
test_mask = drop_7_9(X_test_clean)

X_train_clean = X_train_clean[train_mask]
y_train_clean = y_train_clean[train_mask]

X_test_clean = X_test_clean[test_mask]
y_test_clean = y_test_clean[test_mask]

# 3. Print remaining records
print("X_train_clean shape:", X_train_clean.shape)
print("X_test_clean shape:", X_test_clean.shape)
print("y_train_clean shape:", y_train_clean.shape)
print("y_test_clean shape:", y_test_clean.shape)

# 4. Show percentage of loss of rows
def percent_loss(original, cleaned):
    return 100 * (1 - cleaned.shape[0] / original.shape[0])

print("X_train loss (%):", percent_loss(X_train, X_train_clean))
print("X_test loss (%):", percent_loss(X_test, X_test_clean))
print("y_train loss (%):", percent_loss(y_train, y_train_clean))
print("y_test loss (%):", percent_loss(y_test, y_test_clean))

X_train_clean shape: (729828, 48)
X_test_clean shape: (183464, 48)
y_train_clean shape: (729828,)
y_test_clean shape: (183464,)
X_train loss (%): 60.53296301283413
X_test loss (%): 60.315204530362124
y_train loss (%): 60.53296301283413
y_test loss (%): 60.315204530362124


In [9]:
# For X_train
train_missing_counts = X_train.apply(lambda row: row.isin([7, 9]).sum(), axis=1)
train_missing_table = train_missing_counts.value_counts().sort_index(ascending=False).reset_index()
train_missing_table.columns = ['num_missing_7_9', 'num_records']
train_missing_table['percent_records'] = 100 * train_missing_table['num_records'] / len(X_train)
train_missing_table = train_missing_table.sort_values('num_missing_7_9', ascending=False)
print("X_train missing value distribution:")
print(train_missing_table)

# For X_test
test_missing_counts = X_test.apply(lambda row: row.isin([7, 9]).sum(), axis=1)
test_missing_table = test_missing_counts.value_counts().sort_index(ascending=False).reset_index()
test_missing_table.columns = ['num_missing_7_9', 'num_records']
test_missing_table['percent_records'] = 100 * test_missing_table['num_records'] / len(X_test)
test_missing_table = test_missing_table.sort_values('num_missing_7_9', ascending=False)
print("X_test missing value distribution:")
print(test_missing_table)

# Drop rows with any 7 or 9 in X_train and X_test, and update y_train and y_test accordingly
train_mask = ~train_missing_counts.astype(bool)
test_mask = ~test_missing_counts.astype(bool)

X_train_no_7_9 = X_train[train_mask]
y_train_no_7_9 = y_train[train_mask]

X_test_no_7_9 = X_test[test_mask]
y_test_no_7_9 = y_test[test_mask]

X_train missing value distribution:
    num_missing_7_9  num_records  percent_records
0                31            2         0.000108
1                30            1         0.000054
2                29            2         0.000108
3                28            1         0.000054
4                27            1         0.000054
5                26            5         0.000270
6                25            2         0.000108
7                24           12         0.000649
8                23           13         0.000703
9                22           12         0.000649
10               21           19         0.001027
11               20           30         0.001622
12               19           78         0.004218
13               18          120         0.006489
14               17           79         0.004272
15               16          105         0.005678
16               15          115         0.006219
17               14          258         0.013952
18            

We observe a couple of things, let's deal it case by case- <br>

1. Large number of 7/9 values that don't given us any information.<br>
Ans. Let's encode 7 and 9 to a single value 9 and label it as "Unknown Value"

2. Some occurances of NaN needs to be handled <br>
Ans. Assign it a code of 9 like above since we don't know the value

3. Fix nominal ordering for categorical features like `GENHLTH` and `_RFHLTH` <br>
Ans. Currently the ordering for `GENHLTH` is 1> 2> 3> 4> 5> 7, we need to make it 1 < 2 < 3 < 4 < 5 < 7;
    The ordering for `_RFHLTH` is 1 > 2 > 7, we need to make it 1 < 2 < 7

4. Recode 7 to NaN, since Catboost Natively handles NaN values



(1) Replace 9 with 7 across all categorical values

In [16]:
# Encode categorical features: replace 9 with 7 for all columns in ordinal_features and nomial_features
X_train_no_9 = X_train_no_7_9.copy()
X_test_no_9 = X_test_no_7_9.copy()

for col in ordinal_features + nomial_features:
    if col in X_train_no_9.columns:
        X_train_no_9[col] = X_train_no_9[col].replace(9, 7)
    if col in X_test_no_7_9.columns:
        X_test_no_9[col] = X_test_no_9[col].replace(9, 7)

/var/folders/m2/ycqz5h_d2y55049kl0cst9rm0000gn/T/ipykernel_78470/253456442.py:7: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_train_no_9[col] = X_train_no_9[col].replace(9, 7)
/var/folders/m2/ycqz5h_d2y55049kl0cst9rm0000gn/T/ipykernel_78470/253456442.py:9: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_test_no_9[col] = X_test_no_9[col].replace(9, 7)


Verify if the codes only display 7 for all Categorical columns

In [17]:
# Get category and object columns
cat_obj_features_no_9 = [col for col in X_train_no_9.columns if X_train_no_9[col].dtype.name in ['category', 'object']]

ordinal_features_no_9 = []
nomial_features_no_9 = []

for col in cat_obj_features:
    unique_vals = X_train_no_9[col].unique()
    print(f"{col}: {unique_vals}")

    # Check if the category is ordered
    if X_train_no_9[col].dtype.name == 'category' and X_train_no_9[col].cat.ordered:
        ordinal_features_no_9.append(col)
    else:
        nomial_features_no_9.append(col)

print("Ordinal features:", ordinal_features_no_9)
print("Nomial features:", nomial_features_no_9)

DISPCODE: [1100, 1200]
Categories (2, int64): [1100, 1200]
GENHLTH: [1, 2, 3, 4, 5]
Categories (6, int64): [1 < 2 < 3 < 4 < 5 < 7]
PHYSHLTH: [1, 2, 4, 3]
Categories (4, int64): [1 < 2 < 3 < 4]
MENTHLTH: [1, 2, 4, 3]
Categories (4, int64): [1 < 2 < 3 < 4]
HLTHPLN1: [1, 2]
Categories (3, int64): [1, 2, 7]
PERSDOC2: [1, 3, 2]
Categories (4, int64): [1, 2, 3, 7]
MEDCOST: [2, 1]
Categories (3, int64): [1, 2, 7]
CHECKUP1: [4, 1, 2, 3, NaN]
Categories (4, int64): [1 < 2 < 3 < 4]
CVDINFR4: [2, 1]
Categories (3, int64): [1, 2, 7]
CVDCRHD4: [2, 1]
Categories (3, int64): [1, 2, 7]
CVDSTRK3: [2, 1]
Categories (3, int64): [1, 2, 7]
ASTHMA3: [2, 1]
Categories (3, int64): [1, 2, 7]
CHCSCNCR: [2, 1]
Categories (3, int64): [1, 2, 7]
CHCOCNCR: [2, 1]
Categories (3, int64): [1, 2, 7]
CHCCOPD1: [2, 1]
Categories (3, int64): [1, 2, 7]
HAVARTH3: [2, 1]
Categories (3, int64): [1, 2, 7]
ADDEPEV2: [2, 1]
Categories (3, int64): [1, 2, 7]
CHCKIDNY: [2, 1]
Categories (3, int64): [1, 2, 7]
DIABETE3: [2, 1]
Categor

(2) Deal with NaN in `CHECKUP1` feature

In [18]:
X_train_no_9['CHECKUP1'].value_counts(dropna=False)

CHECKUP1
1      492811
2      100295
4       68068
3       61908
NaN      6746
Name: count, dtype: int64

In [19]:
#Encode NaN in CHECKUP1 with 7
X_train_no_9['CHECKUP1'] = X_train_no_9['CHECKUP1'].cat.add_categories(7).fillna(7)
X_test_no_9['CHECKUP1'] = X_test_no_9['CHECKUP1'].cat.add_categories(7).fillna(7)

Verify if NaN is encoded as 7

In [20]:
X_train_no_9['CHECKUP1'].value_counts(dropna=False)

CHECKUP1
1    492811
2    100295
4     68068
3     61908
7      6746
Name: count, dtype: int64

In [35]:
X_test_no_9['CHECKUP1'].value_counts(dropna=False)

CHECKUP1
1    124192
2     25266
4     16832
3     15487
7      1687
Name: count, dtype: int64

(3) Fix ordinal features [`GENHLTH` and `_RFHLTH`] , fix the order

In [27]:
#display the catergories in list ordinal_features_no_9
for col in ordinal_features_no_9:
    unique_vals = X_train_no_9[col].unique()
    print(f"{col}: {unique_vals}")
    print(f"{col}: {X_train_no_9[col].value_counts()}")


GENHLTH: [1, 2, 3, 4, 5]
Categories (6, int64): [1 < 2 < 3 < 4 < 5 < 7]
GENHLTH: GENHLTH
2    253896
3    212648
1    146579
4     84096
5     32609
7         0
Name: count, dtype: int64
PHYSHLTH: [1, 2, 4, 3]
Categories (4, int64): [1 < 2 < 3 < 4]
PHYSHLTH: PHYSHLTH
1    462169
2    162669
4     66245
3     38745
Name: count, dtype: int64
MENTHLTH: [1, 2, 4, 3]
Categories (4, int64): [1 < 2 < 3 < 4]
MENTHLTH: MENTHLTH
1    464657
2    154625
4     62319
3     48227
Name: count, dtype: int64
CHECKUP1: [4, 1, 2, 3, 7]
Categories (5, int64): [1 < 2 < 3 < 4 < 7]
CHECKUP1: CHECKUP1
1    492811
2    100295
4     68068
3     61908
7      6746
Name: count, dtype: int64
_RFHLTH: [1, 2]
Categories (3, int64): [1 < 2 < 7]
_RFHLTH: _RFHLTH
1    613123
2    116705
7         0
Name: count, dtype: int64
_EDUCAG: [3, 2, 4, 1]
Categories (4, int64): [1 < 2 < 3 < 4]
_EDUCAG: _EDUCAG
4    280257
3    208315
2    192719
1     48537
Name: count, dtype: int64
_INCOMG: [5, 2, 3, 4, 1]
Categories (5, int64):

In [28]:
# Reverse the codes for GENHLTH: 1->5, 2->4, 3->3, 4->2, 5->1, and remove 7 from categories
genhlth_map = {1: 5, 2: 4, 3: 3, 4: 2, 5: 1}

# Apply mapping and remove 7 from categories
for df in [X_train_no_9, X_test_no_9]:
    # Only map codes 1-5, leave 7 as is for unknown
    df['GENHLTH'] = df['GENHLTH'].map(lambda x: genhlth_map.get(x, x))
    # Set new categories and order
    df['GENHLTH'] = df['GENHLTH'].astype('category')
    df['GENHLTH'] = df['GENHLTH'].cat.set_categories([1, 2, 3, 4, 5], ordered=True)

Verify `GENHLTH`, order and value_counts

In [32]:
print(X_train_no_9['GENHLTH'].value_counts(dropna=False))
X_train_no_9['GENHLTH'].unique()

GENHLTH
4    253896
3    212648
5    146579
2     84096
1     32609
Name: count, dtype: int64


[5, 4, 3, 2, 1]
Categories (5, int64): [1 < 2 < 3 < 4 < 5]

Do the same for `_RFHLTH`

In [33]:
# Reverse the codes for _RFHLTH: 1->5, 2->4, 3->3, 4->2, 5->1, and remove 7 from categories
rfhlth_map = {1: 2, 2: 1}

# Apply mapping and remove 7 from categories
for df in [X_train_no_9, X_test_no_9]:
    # Only map codes 1-5, leave 7 as is for unknown
    df['_RFHLTH'] = df['_RFHLTH'].map(lambda x: rfhlth_map.get(x, x))
    # Set new categories and order
    df['_RFHLTH'] = df['_RFHLTH'].astype('category')
    df['_RFHLTH'] = df['_RFHLTH'].cat.set_categories([1, 2], ordered=True)

In [34]:
print(X_train_no_9['_RFHLTH'].value_counts(dropna=False))
X_train_no_9['_RFHLTH'].unique()

_RFHLTH
2    613123
1    116705
Name: count, dtype: int64


[2, 1]
Categories (2, int64): [1 < 2]

Create a copy of the X_train_no_9 and X_test_no_9, to represent codes with actual values to aid final analysis

In [36]:
X_train_for_reporting = X_train_no_9.copy()
X_test_for_reporting = X_test_no_9.copy()

(4) Recode 7 to NaN, (part of feature transformation for conducting SHAP analysis)

In [39]:

# Create new copies for recoding
X_train_with_NaN = X_train_no_9.copy()
X_test_with_NaN = X_test_no_9.copy()

# Recode value 7 in categorical columns to np.nan (unknown)
for col in cat_obj_features_no_9:
    if col in X_train_with_NaN.columns:
        X_train_with_NaN[col] = X_train_with_NaN[col].replace(7, pd.NA)
    if col in X_test_with_NaN.columns:
        X_test_with_NaN[col] = X_test_with_NaN[col].replace(7, pd.NA)


/var/folders/m2/ycqz5h_d2y55049kl0cst9rm0000gn/T/ipykernel_78470/159680260.py:8: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_train_with_NaN[col] = X_train_with_NaN[col].replace(7, pd.NA)
/var/folders/m2/ycqz5h_d2y55049kl0cst9rm0000gn/T/ipykernel_78470/159680260.py:10: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  X_test_with_NaN[col] = X_test_with_NaN[col].replace(7, pd.NA)


In [41]:
#display unique values again to verify
for col in nomial_features_no_9 :
    unique_vals = X_train_with_NaN[col].unique()
    print(f"{col}: {unique_vals}")
    print(f"{col}: {X_train_with_NaN[col].value_counts(dropna=False)}")

DISPCODE: [1100, 1200]
Categories (2, int64): [1100, 1200]
DISPCODE: DISPCODE
1100    696011
1200     33817
Name: count, dtype: int64
HLTHPLN1: [1, 2]
Categories (2, int64): [1, 2]
HLTHPLN1: HLTHPLN1
1    633751
2     96077
Name: count, dtype: int64
PERSDOC2: [1, 3, 2]
Categories (3, int64): [1, 2, 3]
PERSDOC2: PERSDOC2
1    540465
3    137266
2     52097
Name: count, dtype: int64
MEDCOST: [2, 1]
Categories (2, int64): [1, 2]
MEDCOST: MEDCOST
2    624687
1    105141
Name: count, dtype: int64
CVDINFR4: [2, 1]
Categories (2, int64): [1, 2]
CVDINFR4: CVDINFR4
2    706705
1     23123
Name: count, dtype: int64
CVDCRHD4: [2, 1]
Categories (2, int64): [1, 2]
CVDCRHD4: CVDCRHD4
2    705926
1     23902
Name: count, dtype: int64
CVDSTRK3: [2, 1]
Categories (2, int64): [1, 2]
CVDSTRK3: CVDSTRK3
2    712635
1     17193
Name: count, dtype: int64
ASTHMA3: [2, 1]
Categories (2, int64): [1, 2]
ASTHMA3: ASTHMA3
2    629259
1    100569
Name: count, dtype: int64
CHCSCNCR: [2, 1]
Categories (2, int64): [1

Let's check all teh varialbes in our datasets one last time before moving onto SHAP analysis

In [42]:
#combine X_train_with_NaN and X_test_with_NaN into a single dataframe for reporting
X_combined_with_NaN = pd.concat([X_train_with_NaN, X_test_with_NaN], axis=0)

In [43]:
X_combined_with_NaN.info()

<class 'pandas.core.frame.DataFrame'>
Index: 913292 entries, 1395283 to 1321228
Data columns (total 48 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   DISPCODE  913292 non-null  category
 1   GENHLTH   913292 non-null  category
 2   PHYSHLTH  913292 non-null  category
 3   MENTHLTH  913292 non-null  category
 4   HLTHPLN1  913292 non-null  category
 5   PERSDOC2  913292 non-null  category
 6   MEDCOST   913292 non-null  category
 7   CHECKUP1  904859 non-null  category
 8   CVDINFR4  913292 non-null  category
 9   CVDCRHD4  913292 non-null  category
 10  CVDSTRK3  913292 non-null  category
 11  ASTHMA3   913292 non-null  category
 12  CHCSCNCR  913292 non-null  category
 13  CHCOCNCR  913292 non-null  category
 14  CHCCOPD1  913292 non-null  category
 15  HAVARTH3  913292 non-null  category
 16  ADDEPEV2  913292 non-null  category
 17  CHCKIDNY  913292 non-null  category
 18  DIABETE3  913292 non-null  category
 19  SEX       913292 non-

In [44]:
X_combined_with_NaN.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
DISPCODE,913292.0,2.0,1100.0,870798.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
GENHLTH,913292.0,5.0,4.0,317231.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PHYSHLTH,913292.0,4.0,1.0,578185.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MENTHLTH,913292.0,4.0,1.0,581420.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
HLTHPLN1,913292.0,2.0,1.0,792986.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PERSDOC2,913292.0,3.0,1.0,676468.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MEDCOST,913292.0,2.0,2.0,781921.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CHECKUP1,904859.0,4.0,1.0,617003.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CVDINFR4,913292.0,2.0,2.0,884491.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CVDCRHD4,913292.0,2.0,2.0,883418.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
